# TTT-Linear Inner-State Inspection / TTT-Linear 内层状态检查

Runs a TTT-Linear layer on a synthetic sequence and inspects how the inner weight matrix `W` evolves over mini-batch segments. Useful for:
- Sanity-checking the delta-rule update behaves smoothly
- Understanding the effect of `eta` and `mini_batch` on inner dynamics
- Sanity-checking Triton parity when that backend arrives (Stage 2b)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import math
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.models.ttt_linear import ttt_linear_forward_pytorch

In [ ]:
# ---- Config ----
torch.manual_seed(0)
B, T, d = 1, 32, 8
x = torch.randn(B, T, d) * 0.5
tK = torch.randn(d, d) * 0.3
tV = torch.randn(d, d) * 0.3
tQ = torch.randn(d, d) * 0.3

y, W_final = ttt_linear_forward_pytorch(x, tK, tV, tQ, torch.tensor(0.02), mini_batch=4)
print('y shape:', y.shape)
print('W_final shape:', W_final.shape)
print('W_final norm:', W_final.norm().item())

In [ ]:
# ---- Manually replay to capture per-segment W snapshots ----
# (Same computation as ttt_linear_forward_pytorch but we retain W at every seg boundary.)

def replay_with_snapshots(x, tK, tV, tQ, eta_val, mini_batch):
    K = x @ tK; V = x @ tV; Q = x @ tQ
    B, T, d = K.shape
    W = torch.zeros(B, d, d)
    snaps = [W.clone()]
    n_seg = math.ceil(T / mini_batch)
    for seg in range(n_seg):
        t0 = seg * mini_batch
        t1 = min(t0 + mini_batch, T)
        K_seg = K[:, t0:t1, :]
        V_seg = V[:, t0:t1, :]
        Wk = K_seg @ W.transpose(-1, -2)
        residual = Wk - V_seg
        grad = residual.transpose(-1, -2) @ K_seg
        W = W - eta_val * grad
        snaps.append(W.clone())
    return snaps

snaps = replay_with_snapshots(x, tK, tV, tQ, 0.02, 4)
norms = np.array([s.norm().item() for s in snaps])
print('per-segment W norms:', norms.round(3))

In [ ]:
# ---- Plot the growth of ||W|| segment by segment ----
plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(norms)), norms, marker='o')
plt.xlabel('segment index')
plt.ylabel('||W|| (Frobenius)')
plt.title('TTT-Linear inner weight norm across segments')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Sweep over mini_batch to see effect on final ||W|| ----
final_norms = {}
for mb in [1, 2, 4, 8, 16, 32]:
    _, W = ttt_linear_forward_pytorch(x, tK, tV, tQ, torch.tensor(0.02), mini_batch=mb)
    final_norms[mb] = W.norm().item()
print(final_norms)

plt.figure(figsize=(6, 4))
plt.plot(list(final_norms.keys()), list(final_norms.values()), marker='s')
plt.xscale('log', base=2)
plt.xlabel('mini_batch (log₂ scale)')
plt.ylabel('final ||W||')
plt.title('Effect of mini-batch size on final inner weight')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()